In [ ]:
!pip install -U sentence-transformers torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 661.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import random
import json
import torch
import numpy as np
from sentence_transformers import SentenceTransformer,InputExample,losses
from torch.utils.data import DataLoader

In [2]:
#load data set for training
train_data = []

with open("/content/synthetic_data_for_classification (1).jsonl", "r", encoding="utf-8") as f:
    for line in f:
        train_data.append(json.loads(line))



FileNotFoundError: [Errno 2] No such file or directory: '/content/synthetic_data_for_classification (1).jsonl'

In [ ]:
#light preprocessing
def clean_text(text):
  text=text.lower()
  text=" ".join(text.split())
  return text

In [ ]:
train_examples = []

for item in train_data:
    # Skip broken rows safely
    if item["anchor_text"] is None or item["text_a"] is None or item["text_b"] is None:
        continue

    anchor = clean_text(item["anchor_text"])
    a = clean_text(item["text_a"])
    b = clean_text(item["text_b"])

    if item["text_a_is_closer"] is True:
        positive, negative = a, b
    else:
        positive, negative = b, a

    train_examples.append(
        InputExample(texts=[anchor, positive, negative])
    )


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

#defining triplet loss
train_loss=losses.TripletLoss(
    model=model,
    distance_metric=losses.TripletDistanceMetric.COSINE,
    triplet_margin=0.3
    #A margin of 0.3 enforces a meaningful separation between similar and dissimilar narratives while remaining stable for training on noisy narrative similarity data.”
)

#dataloader
train_dataloader=DataLoader(
    train_examples,
    shuffle=True,
    batch_size=16
)

In [ ]:
#finetuning
model.fit(
    train_objectives=[(train_dataloader,train_loss)],
    epochs=1,  #prevents overfitting
    warmup_steps=100,
    show_progress_bar=True

)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: You chose "Don't visualize my results"


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


In [1]:
#encode track-b data
with open("/content/dev_track_b.jsonl", "r", encoding="utf-8") as f:
    trackB_dev = json.load(f)

dev_texts = [clean_text(item["text"]) for item in trackB_dev]

#convert to embedings
dev_embeddings = model.encode(
    dev_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)



NameError: name 'json' is not defined

In [ ]:
with open("trackB_test.json", "r", encoding="utf-8") as f:
    trackB_test = json.load(f)

test_texts = [clean_text(item["text"]) for item in trackB_test]

test_embeddings = model.encode(
    test_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

np.save("trackB_submission_embeddings.npy", test_embeddings)
